# Databricks SQL Week — Day 3
## Joins, Set Operations & Multi-Table Query Patterns
 
**Date:** 27 May 2026  

---

### Our Schema Setup
Before writing queries, make sure you understand the available tables created in the **Setup Notebook**:
- `users`: Stores all profiles (role = `admin` or `customer`)
- `orders`: Tracks actual transactions (`user_id`, `product_id`, `quantity`, `total_amount`)
- `products`: The product catalog (`id`, `category_id`, `name`, `price`, `sku`)
- `categories`: Hierarchical categories (`id`, `name`, `parent_id` for nested subcategories)
- `inventory`: Real-time stock counts (`product_id`, `quantity`, `reorder_level`)

Let's switch to our active schema database to begin our work!

In [0]:
USE CATALOG workshop;
USE SCHEMA ecommerce_sql;

-- Check all the tables
SHOW TABLES;

## Story: 
You are a data analyst at ShopElite, an e-commerce platform. Your mission is to help different teams (Marketing, Inventory, Growth) make data-driven decisions by writing SQL queries that connect and analyze information across multiple tables. You'll use JOINs to answer real business questions, such as identifying active buyers for campaigns, checking stock levels, and finding customers who haven't made a purchase yet.

---

# 1. Why Joins Are Important & Key Relational Concepts

### The Explanation
Relational databases are structured around **keys** to link tables together:
- **Primary Key (PK):** A unique identifier for a row (e.g., `users.id`).
- **Foreign Key (FK):** A reference in one table that links to the primary key of another table (e.g., `orders.user_id` pointing to `users.id`).


### Cardinality Relationships
1. **One-to-One (1:1):** Each product in `products` has exactly one corresponding row in `inventory` (`products.id` <-> `inventory.product_id`).
2. **One-to-Many (1:N):** One customer in `users` can place many orders in `orders` (`users.id` <-> `orders.user_id`).
3. **Many-to-Many (N:M):** Multiple users can buy the same product, and a single product can be purchased by multiple users. We resolve this in e-commerce using a junction/transaction table like `orders` to link `users` and `products` together.


### Alias  
An *alias* is a temporary name given to a table or column for the duration of a query. It makes results easier to read and queries easier to write, especially when joining tables or renaming columns in the output. Use the `AS` keyword (optional in most SQL dialects).

**Example:**  
sql
```
SELECT u.first_name AS user_first, o.total_amount AS order_value
FROM users u
JOIN orders o ON u.id = o.user_id;
```

# 2. INNER JOIN

### The Explanation
An `INNER JOIN` returns rows **only** when there is a matching key in both the left and right tables. If a key exists on the left but not on the right (or vice-versa), that row is completely excluded from the result.

| Left Table (Users) | Right Table (Orders)         | Result      |
|--------------------|-----------------------------|-------------|
| User_ID: 2001      | No Orders Exist             | Excluded    |
| User_ID: 2002      | Order Placed                | Included    |

### Question 1

**Scenario:**  
The **Marketing Team** wants to launch a premium VIP email campaign. However, they only want to target registered users who have actually placed at least one purchase order, so they don't waste budget on inactive accounts.


In [0]:
-- Find all registered users who have placed orders along with order details
SELECT 
    u.id AS user_id,
    u.first_name,
    u.last_name,
    o.id AS order_id,
    o.total_amount,
    o.status
FROM users u
INNER JOIN orders o 
    ON u.id = o.user_id
ORDER BY o.total_amount DESC;

### Real-World Challenge
**Scenario:** The inventory team needs to inspect stock levels. They want a report containing the product name, its price, and the exact quantity available in stock. Because they only care about products currently tracked in the inventory system.

**Your Task:** Write an SQL query using an `INNER JOIN` between `products` and `inventory` that returns the product name, price, SKU, and physical stock quantity.

In [0]:
-- TODO: Write your query below



<details>
<summary>💡 Click to View Solution</summary>

```sql
SELECT 
    p.name AS product_name,
    p.price,
    p.sku,
    i.quantity AS stock_qty
FROM products p
INNER JOIN inventory i 
    ON p.id = i.product_id;
```
</details>

---

# 3. LEFT JOIN (LEFT OUTER JOIN)

### The Explanation
A `LEFT JOIN` returns **all rows** from the left table, and the matching rows from the right table. If there is no match on the right side, the result columns for the right table will contain `NULL` values.

```
   [Left Table]           [Right Table]
   (Users)                (Orders)
   User_ID: 2001   =====> No Orders -> (Returns NULLs for Order fields)
   User_ID: 2002   =====> Order 5001 -> (Returns Order details)
```

### The ShopElite Story
The **Growth Marketing Team** is worried about customer conversion. They want to identify registered customers who signed up but have **never** placed a single order. They want to email these users a personalized "Welcome 50% Off" discount coupon to incentivize their first purchase.

### Code Example

In [0]:
-- Show all users alongside their orders. Users with no orders show NULL for order fields.
SELECT 
    u.id AS user_id,
    u.email,
    u.first_name,
    o.id AS order_id,
    o.total_amount
FROM users u
LEFT JOIN orders o 
    ON u.id = o.user_id
ORDER BY o.id ASC;

### Real-World Challenge
**Scenario:** Identify the exact list of inactive customer accounts who have placed zero orders. 

**Your Task:** Write a query that joins `users` and `orders` using a `LEFT JOIN`, filters specifically for accounts that have no orders, and displays their unique ID, name, and email address.

In [0]:
-- TODO: Write your query below



<details>
<summary>Click to View Solution</summary>

```sql
SELECT 
    u.id AS user_id,
    u.first_name,
    u.last_name,
    u.email
FROM users u
LEFT JOIN orders o 
    ON u.id = o.user_id
WHERE o.id IS NULL
  AND u.role = 'customer';
```
</details>

---

# 4. RIGHT JOIN & FULL OUTER JOIN

### The Explanation
- **RIGHT JOIN:** Returns all rows from the right table, and matching rows from the left. (Note: In practice, professionals prefer using `LEFT JOIN` by switching table positions to keep queries readable, but `RIGHT JOIN` is useful to know).
- **FULL OUTER JOIN:** Returns all rows from **both** tables. If a row has a match on one side but not the other, `NULL` values are filled in for the missing side. This is extremely useful for **data reconciliation** and system integrity audits.

### The ShopElite Story
The **Finance Auditing Team** needs to perform a data integrity check. They want to check for data anomalies, such as:
1. Orders placed by non-existent users (orphaned transactions).
2. Users who exist but have no sales orders.

### Code Example

In [0]:
-- Audit check combining all users and all orders to locate orphaned records on either side
SELECT 
    u.id AS user_id,
    u.email AS user_email,
    o.id AS order_id,
    o.total_amount
FROM users u
FULL OUTER JOIN orders o 
    ON u.id = o.user_id;

### Real-World Challenge
**Scenario:** A bug was reported in the catalog synchronization script. Marketing suspects some products might have been cataloged in `products` but are completely missing real-time stock lines in `inventory`.

**Your Task:** Write an audit query using a `FULL OUTER JOIN` or `RIGHT JOIN` between `products` and `inventory` to find any products that have no matching row in the `inventory` table.

In [0]:
-- TODO: Write your query below



<details>
<summary>Click to View Solution</summary>

```sql
SELECT 
    p.id AS product_id,
    p.name AS product_name,
    i.id AS inventory_id,
    i.quantity
FROM products p
FULL OUTER JOIN inventory i 
    ON p.id = i.product_id
WHERE i.product_id IS NULL OR p.id IS NULL;
```
</details>

---

# 5. CROSS JOIN

### The Explanation
A `CROSS JOIN` creates a **Cartesian Product**. It matches every single row of Table A with every single row of Table B. If Table A has *X* rows and Table B has *Y* rows, the output will contain *X × Y* rows.

| Table A | Table B | Output      |
|---------|---------|-------------|
| Red     | Circle  | Red Circle  |
| Red     | Square  | Red Square  |
| Blue    | Circle  | Blue Circle |
| Blue    | Square  | Blue Square |

> [!WARNING]
> **Performance Warning:** Running a `CROSS JOIN` on large tables (e.g., 100,000 rows x 100,000 rows) generates 10 billion rows, which can freeze or crash your database cluster. Use carefully and always filter or run on small lists.

### The ShopElite Story
The **Logistics Operations Director** wants to construct a lookup matrix containing every product crossed with every distinct state our customers live in. This matrix will be populated with default shipping rules and delivery zone estimates.

### Code Example

In [0]:
-- Cross product of active premium products and customer shipping states
WITH active_states AS (
    SELECT DISTINCT shipping_address AS state 
    FROM orders 
    WHERE shipping_address IS NOT NULL
),
premium_products AS (
    SELECT name, price 
    FROM products 
    WHERE price > 100000.00
)
SELECT 
    p.name AS product_name,
    p.price,
    s.state AS target_state
FROM premium_products p
CROSS JOIN active_states s
ORDER BY product_name, target_state;

### Real-World Challenge
**Scenario:** The Marketing team wants to run a focus test group pairing all registered admin users with all category names to draft localized curation guidelines.

**Your Task:** Write a query that performs a `CROSS JOIN` between `users` (filtered for role = `admin`) and the list of categories, displaying the admin name and category name.

In [0]:
-- TODO: Write your query below



<details>
<summary>Click to View Solution</summary>

```sql
SELECT 
    u.first_name,
    u.last_name,
    c.name AS category_name
FROM users u
CROSS JOIN categories c
WHERE u.role = 'admin';
```
</details>

---

# 6. SELF JOIN

### The Explanation
A `SELF JOIN` is a standard join where a table is **joined to itself**. This requires referencing the table twice using different aliases (e.g., `Table A` and `Table B`) to avoid catalog lookup collisions.

This is highly used for **hierarchical data** (Employee -> Manager, Category -> Parent Category, or finding duplicate records).

### The ShopElite Story
The **SEO and Catalog Navigation Team** is building category breadcrumbs for the navigation header. The `categories` table is recursive: each subcategory has a `parent_id` referencing a main category's primary `id`.

```
   [categories c1 (Subcategory)]   JOIN   [categories c2 (Parent)]
   Name: Laptops                         Name: Electronics
   parent_id: 1001 ---------------------> id: 1001
```

### Code Example

In [0]:
-- Self Join to map subcategories to their corresponding parent categories
SELECT 
    sub.name AS subcategory_name,
    sub.slug AS subcategory_slug,
    parent.name AS parent_category_name
FROM categories sub
INNER JOIN categories parent 
    ON sub.parent_id = parent.id
ORDER BY parent_category_name, subcategory_name;

### Real-World Challenge
**Scenario:** Identify all the top-level "root" categories (those with no parent category) and show their subcategories, using a `LEFT JOIN` on itself so you don't exclude root categories that don't have children yet.

**Your Task:** Write a self-join query that displays the parent category name and its child subcategory name. Include parents even if they do not have children.

In [0]:
-- TODO: Write your query below



<details>
<summary>Click to View Solution</summary>

```sql
SELECT 
    parent.name AS parent_category,
    child.name AS subcategory
FROM categories parent
LEFT JOIN categories child 
    ON child.parent_id = parent.id
WHERE parent.parent_id IS NULL
ORDER BY parent_category;
```
</details>

---

# 7. Multi-Condition JOIN

### The Explanation
Sometimes joining on a single key isn't enough. You can add multiple evaluation criteria in your `ON` statement using `AND` operators. This is common when joining on **composite keys** or matching records within **date ranges** or financial boundaries.

### The ShopElite Story
The **Flash Promo Team** launched a special promotion. They want to audit all orders for premium products and filter out orders where the billing address and shipping address did not match, as this is a signal of potential reseller activity. They also want to ensure they only inspect orders that are active (`delivered` or `shipped`).

### Code Example

In [0]:
-- Join orders to products and users, applying multiple criteria directly within the ON clause
SELECT 
    o.id AS order_id,
    u.first_name,
    u.last_name,
    p.name AS product_name,
    o.total_amount
FROM orders o
JOIN users u 
    ON o.user_id = u.id
   AND o.shipping_address = o.billing_address
JOIN products p 
    ON o.product_id = p.id
   AND p.is_active = true
WHERE o.status IN ('delivered', 'shipped');

### Real-World Challenge
**Scenario:** The Loyalty Rewards team wants to identify orders where customers bought high-value products (price > 50,000) and ordered a quantity of exactly 1 item.

**Your Task:** Join `orders` and `products` using multiple join conditions: join by product ID, and require that the product price is above 50,000 directly within the join clause. Filter out cancelled orders.

In [0]:
-- TODO: Write your query below



<details>
<summary>Click to View Solution</summary>

```sql
SELECT 
    o.id AS order_id,
    p.name AS product_name,
    p.price,
    o.quantity
FROM orders o
INNER JOIN products p 
    ON o.product_id = p.id
   AND p.price > 50000.00
WHERE o.status != 'cancelled';
```
</details>

---

# 8. Semi-Joins via EXISTS

### The Explanation
A **Semi-Join** checks whether a record exists in another table, but **does not append any columns** from that table. In SQL, this is performed using the `EXISTS` keyword inside a `WHERE` clause.

Why use `EXISTS` instead of `INNER JOIN`?
1. **No Duplicates:** Because it just checks for existence, it never multiplies rows from Table A, even if Table B has thousands of matches.
2. **Speed:** The SQL engine halts searching as soon as the first match is located (early exit evaluation).

### The ShopElite Story
The **Logistics Delivery Team** needs to compile a clean, unique list of registered users who have placed at least one purchase order. If you write an `INNER JOIN`, users who have placed 50 orders will appear 50 times, forcing you to write a heavy `DISTINCT` query. Using `EXISTS` keeps the list unique naturally.

### Code Example

In [0]:
-- Get a clean list of unique users who have made purchases using EXISTS
SELECT 
    u.id AS user_id,
    u.first_name,
    u.last_name,
    u.email
FROM users u
WHERE EXISTS (
    SELECT 1 
    FROM orders o 
    WHERE o.user_id = u.id
);

### Real-World Challenge
**Scenario:** The Catalog Management Team wants a list of category names that contain at least one cataloged product, avoiding any category row duplication.

**Your Task:** Write a query using `EXISTS` to find all unique categories that have products associated with them.

In [0]:
-- TODO: Write your query below



<details>
<summary>Click to View Solution</summary>

```sql
SELECT 
    c.id AS category_id,
    c.name AS category_name
FROM categories c
WHERE EXISTS (
    SELECT 1 
    FROM products p 
    WHERE p.category_id = c.id
);
```
</details>

---

# 9. Anti-Joins via NOT EXISTS

### The Explanation
An **Anti-Join** finds records in Table A that **do not exist** in Table B. In SQL, this is performed using the `NOT EXISTS` operator. 

This is highly used for **churn tracking, system alerts, and campaign exclusions**.

> [!IMPORTANT]
> **Why NOT EXISTS is preferred over NOT IN:** If the subquery column inside a `NOT IN` filter contains even a single `NULL` value, the entire query returns **zero rows**. `NOT EXISTS` is completely NULL-safe and much faster on large datasets.

### The ShopElite Story
The **Retention Marketing Team** wants to run a re-engagement campaign. They want to retrieve a clean, unique list of customers who have **never** made a purchase, so they can send them a massive discount. 

### Code Example

In [0]:
-- Retrieve all customer users who have never placed a single order using NOT EXISTS
SELECT 
    u.id AS user_id,
    u.first_name,
    u.last_name,
    u.email
FROM users u
WHERE u.role = 'customer'
  AND NOT EXISTS (
      SELECT 1 
      FROM orders o 
      WHERE o.user_id = u.id
  );

### Real-World Challenge
**Scenario:** A stock auditing alarm needs to notify the warehouse team about cataloged products that have absolutely no inventory entry whatsoever.

**Your Task:** Write a query using `NOT EXISTS` that retrieves products that do not exist in the `inventory` table.

In [0]:
-- TODO: Write your query below



<details>
<summary>Click to View Solution</summary>

```sql
SELECT 
    p.id AS product_id,
    p.name AS product_name,
    p.sku
FROM products p
WHERE NOT EXISTS (
    SELECT 1 
    FROM inventory i 
    WHERE i.product_id = p.id
);
```
</details>

---

# 10. The Join Fanout Problem

### The Explanation
**Join Fanout** occurs when a join unintentionally duplicates rows, multiplying your calculations. This happens when you perform a **One-to-Many** or **Many-to-Many** join and then try to run aggregations (like `SUM` or `COUNT`) without accounting for row multiplication.

### The ShopElite Story
The **VP of Finance** is furious. A junior analyst reported that ShopElite generated a massive total revenue in a single query. However, the bank statement has only half of that amount! The analyst joined `users` to `orders` and did `SUM(total_amount)`, but didn't realize that joining duplicated some metrics due to fanout.

Let's see what happened.

In [0]:
-- NAIVE QUERY causing fanout! Run this and look at the counts.
SELECT 
    COUNT(u.id) AS row_count,
    SUM(o.total_amount) AS calculated_revenue
FROM users u
LEFT JOIN orders o 
    ON u.id = o.user_id;

### The Fix: Aggregate Before Joining!
To solve fanout, you must **pre-aggregate** your metrics in a subquery or a Common Table Expression (CTE) before joining the tables together. This guarantees that you are joining exactly **one row per key**.

In [0]:
-- CORRECT QUERY: Pre-aggregate orders to get 1 row per user before joining!
WITH user_sales AS (
    SELECT 
        user_id,
        SUM(total_amount) AS actual_revenue
    FROM orders
    GROUP BY user_id
)
SELECT 
    u.id AS user_id,
    u.first_name,
    COALESCE(us.actual_revenue, 0) AS total_revenue
FROM users u
LEFT JOIN user_sales us 
    ON u.id = us.user_id
ORDER BY total_revenue DESC;

### Real-World Challenge
**Scenario:** Calculate the total stock quantity per category. If you join `categories` $\rightarrow$ `products` $\rightarrow$ `inventory` and calculate `SUM(inventory.quantity)`, you will get incorrect values if a product is cataloged twice or if keys duplicate rows.

**Your Task:** Write a query that pre-aggregates the `inventory` table by `product_id` (or uses clean aggregation) to return the total stock count per category name.

In [0]:
-- TODO: Write your query below



<details>
<summary>Click to View Solution</summary>

```sql
WITH product_stock AS (
    SELECT 
        product_id,
        SUM(quantity) AS total_qty
    FROM inventory
    GROUP BY product_id
)
SELECT 
    c.name AS category_name,
    SUM(COALESCE(ps.total_qty, 0)) AS category_stock_total
FROM categories c
JOIN products p 
    ON c.id = p.category_id
LEFT JOIN product_stock ps 
    ON p.id = ps.product_id
GROUP BY c.name
ORDER BY category_stock_total DESC;
```
</details>

---

# 11. Set Operations

### The Explanation
While joins combine columns from different tables side-by-side, **Set Operations** combine **rows** from different queries vertically.

```
   [Join: Horizontal]                [Set Operation: Vertical]
   Table A  |  Table B                Table A (Row 1)
   ------------------                Table A (Row 2)
   Col A    |  Col B                  ===============
                                      Table B (Row 1)
```

### The Four Operations
1. **UNION ALL:** Combines row outputs, keeping all duplicates (fastest, highly preferred in production).
2. **UNION:** Combines and deduplicates row outputs (adds an expensive sort/unique step in Spark).
3. **INTERSECT:** Returns only rows that appear in **both** query outputs.
4. **EXCEPT (or MINUS):** Returns rows from the first query that are **not** present in the second query.

> [current_timestamp()]: **Rules for Set Operations:** Both queries must return the **exact same number of columns**, in the **same order**, with **compatible data types**.

### The ShopElite Story
Marketing wants a combined targeted user cohort consisting of users from Delhi/Mumbai (Region A) and users who bought premium laptops (Region B). They need vertical row stacking.

### Code Example

In [0]:
-- Combine high-value transactions with transactions shipping to Maharashtra
SELECT id AS order_id, user_id, total_amount, 'High Value' AS segment
FROM orders
WHERE total_amount > 50000.00

UNION ALL

SELECT id AS order_id, user_id, total_amount, 'Maharashtra Region' AS segment
FROM orders
WHERE shipping_address LIKE '%Maharashtra%';

### Real-World Challenge
**Scenario:** Find the unique customer user IDs who placed orders in *both* `delivered` status *and* `shipped` status (active engagement).

**Your Task:** Write a query using `INTERSECT` to identify the customer IDs present in both order status groups.

In [0]:
-- TODO: Write your query below



<details>
<summary>Click to View Solution</summary>

```sql
SELECT user_id 
FROM orders 
WHERE status = 'delivered'

INTERSECT

SELECT user_id 
FROM orders 
WHERE status = 'shipped';
```
</details>

---

# 12. The Date Spine Pattern (Industry Standard)

### The Explanation
In business intelligence dashboards, you want to show a trend line of daily revenue. However, if a retail store has **zero sales** on Tuesday, standard grouping completely omits Tuesday from the results. The trend line skips from Monday to Wednesday, hiding a critical operational detail: zero sales.

To solve this, analytics engineers build a **Date Spine** — a continuous calendar list of dates — and then `LEFT JOIN` the transaction data onto that spine, replacing empty sales values with `0`.

### The ShopElite Story
The executive leadership team is panicking because the weekly sales dashboard has missing dates, making it look like the system is offline. You need to construct a continuous calendar trend line for May 2026 using Databricks native SQL functions.

### Code Example

In [0]:
-- Generate a continuous 31-day date spine for May 2026
WITH date_spine AS (
    SELECT 
        EXPLODE(
            SEQUENCE(
                TO_DATE('2026-05-01'), 
                TO_DATE('2026-05-31'), 
                INTERVAL 1 DAY
            )
        ) AS calendar_date
)
SELECT calendar_date 
FROM date_spine
ORDER BY calendar_date ASC;

Let's LEFT JOIN our live orders onto this date spine to report continuous revenue, substituting NULLs with `0` using the `COALESCE` function.

In [0]:
WITH date_spine AS (
    SELECT 
        EXPLODE(
            SEQUENCE(
                TO_DATE('2026-05-01'), 
                TO_DATE('2026-05-31'), 
                INTERVAL 1 DAY
            )
        ) AS calendar_date
),
daily_revenue AS (
    SELECT 
        TO_DATE(created_at) AS order_day,
        SUM(total_amount) AS revenue
    FROM orders
    GROUP BY TO_DATE(created_at)
)
SELECT 
    ds.calendar_date,
    COALESCE(dr.revenue, 0.00) AS daily_revenue
FROM date_spine ds
LEFT JOIN daily_revenue dr 
    ON ds.calendar_date = dr.order_day
ORDER BY ds.calendar_date ASC;

### Real-World Challenge
**Scenario:** The VP of Operations wants to track the count of daily orders placed in the catalog for the first 10 days of May 2026. If a day has no orders, they want to see `0` orders instead of having the date missing.

**Your Task:** Generate a date spine for `2026-05-01` to `2026-05-10` and `LEFT JOIN` orders onto it, displaying the calendar date and the count of orders.

In [0]:
-- TODO: Write your query below



<details>
<summary>Click to View Solution</summary>

```sql
WITH date_spine AS (
    SELECT 
        EXPLODE(
            SEQUENCE(
                TO_DATE('2026-05-01'), 
                TO_DATE('2026-05-10'), 
                INTERVAL 1 DAY
            )
        ) AS calendar_date
),
daily_orders AS (
    SELECT 
        TO_DATE(created_at) AS order_day,
        COUNT(id) AS order_count
    FROM orders
    GROUP BY TO_DATE(created_at)
)
SELECT 
    ds.calendar_date,
    COALESCE(do.order_count, 0) AS total_orders
FROM date_spine ds
LEFT JOIN daily_orders do 
    ON ds.calendar_date = do.order_day
ORDER BY ds.calendar_date ASC;
```
</details>

---

# 13. Capstone Practice Questions

Now it is time to put your analytics engineering skills to the test with these comprehensive challenges using our live `ShopElite` e-commerce schema.

### Beginner Challenge
**Task:** Retrieve all products (`products`) along with their parent category name (`categories.name`). If a product belongs to a category that has no parent, still display the product.

In [0]:
-- TODO: Write your query below



<details>
<summary>Click to View Solution</summary>

```sql
SELECT 
    p.name AS product_name,
    c.name AS category_name
FROM products p
LEFT JOIN categories c 
    ON p.category_id = c.id;
```
</details>

---

### Intermediate Challenge
**Task:** Find the total purchase revenue generated by each customer user role (`users.role`). Show the role and its total revenue. Exclude users who are admins, and display roles even if they spent `0` rupees (substituting NULLs with `0.00`).

In [0]:
-- TODO: Write your query below



<details>
<summary>Click to View Solution</summary>

```sql
WITH sales_by_user AS (
    SELECT 
        user_id,
        SUM(total_amount) AS user_spend
    FROM orders
    GROUP BY user_id
)
SELECT 
    u.role,
    SUM(COALESCE(s.user_spend, 0.00)) AS total_role_spend
FROM users u
LEFT JOIN sales_by_user s 
    ON u.id = s.user_id
WHERE u.role != 'admin'
GROUP BY u.role;
```
</details>

---

### Advanced Challenge
**Task:** Churn Detection and Activity Spine. Identify customers (role = `customer`) who have **never** placed a high-value order (total amount > 30,000) or whose orders are entirely empty, using a `NOT EXISTS` anti-join pattern for high-performance scale.

In [0]:
-- TODO: Write your query below



<details>
<summary>Click to View Solution</summary>

```sql
SELECT 
    u.id AS user_id,
    u.first_name,
    u.last_name,
    u.email
FROM users u
WHERE u.role = 'customer'
  AND NOT EXISTS (
      SELECT 1 
      FROM orders o 
      WHERE o.user_id = u.id
        AND o.total_amount > 30000.00
  );
```
</details>

---